# Documentation

pymupdf


This notebook demonstrates how to extract transaction data from a PDF bank statement, clean it, and transform it into a standardized JSON format. Below is a breakdown of each step:



In [ ]:
https://pypi.org/project/pdfplumber/

# Installation

pdfplumber: A Python library for extracting text and tables from PDFs.This works best on machine-generated, rather than scanned PDFs

pandas: Used for data manipulation and transformation.

Installs required libraries:

In [ ]:
!pip install pdfplumber
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 38.0 MB/s eta 0:00:00


# EXPECTED FORMAT

The goal is to structure transactions into a JSON array of objects with fields:
`id`, `type`, `amount`, `narration`, `date`, `balance`.

Example:



```
[
  {
    "id": "0",
    "type": "credit",
    "amount": 17880,
    "narration": "Fixed Deposit Closure Proceeds - Principal",
    "date": "2020-05-18T00:00:00",
    "balance": 18170
  },
  {
    "id": "0",
    "type": "debit",
    "amount": 17880,
    "narration": "ALAT NIP TRANSFER TO POLARIS-tithe",
    "date": "2020-05-18T00:00:00",
    "balance": 290
  },
  {
    "id": "0",
    "type": "debit",
    "amount": 25,
    "narration": "COMM ALAT NIP TRANSFER TO POLARIS-tithe",
    "date": "2020-05-18T00:00:00",
    "balance": 265
  }
]
```



# IMPORTS

In [ ]:
import pdfplumber
import pandas as pd

# FUNCTION TO EXTRACT

You can find table extraction guild following this link: ` https://github.com/jsvine/pdfplumber`

In [ ]:
def extract_and_concat_tables_from_pdf(pdf_path):
    """
    Extracts tables from a PDF using pdfplumber and concatenates them into a single Pandas DataFrame.

    Args:
        pdf_path (str): Path to the PDF file.

    Returns:
        pd.DataFrame: A single concatenated DataFrame containing all extracted tables.
    """
    tables = []  # List to store extracted tables
    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            print(f"Processing page {page_number}...")
            page_tables = page.extract_tables(
                {
                "vertical_strategy": "lines",  # Use lines to detect vertical separators
                "horizontal_strategy": "lines",  # Use lines to detect horizontal separators
                "intersection_tolerance": 5,  # Adjust sensitivity
                }


            )  # Extract tables from the page
            for table in page_tables:
                # Convert the table to a Pandas DataFrame
                df = pd.DataFrame(table)
                tables.append(df)

    # Concatenate all DataFrames into a single DataFrame
    if tables:
        concatenated_df = pd.concat(tables, ignore_index=True)
        return concatenated_df
    else:
        print("No tables were found in the PDF.")
        return pd.DataFrame()

LOADING THE PDF

In [ ]:
pdf_path = '/content/your_bank_statement.pdf'
gtb = extract_and_concat_tables_from_pdf(pdf_path)

CHECK THE EXTRACTED DATA

In [ ]:
gtb.head(50)

START THE CLEANING PROCESS

---



# EXAMPLE 1- ALAT


In [ ]:
pdf_path = '/content/27052020115935_Statement_For_OLUWASOGO OGUNDOWOLE_0234952225_9993 (1).pdf'
alat = extract_and_concat_tables_from_pdf(pdf_path)

Processing page 1...
Processing page 2...


Processing page 3...


In [ ]:
alat

,0,1,2,3,4,5,6
0,DATE,REFERENCE NUMBER,TRANSACTION DETAILS,Debit,Credit,Balance,NaN
1,18-May-2020,M4439,Fixed Deposit Closure Proceeds - Principal,,"17,880.00","18,170.48",NaN
2,18-May-2020,S5741873,ALAT NIP TRANSFER TO POLARIS-tithe,"17,880.00",,290.48,NaN
3,18-May-2020,S5741873,COMM ALAT NIP TRANSFER TO POLARIS-tithe,25.00,,265.48,NaN
4,18-May-2020,S5741873,VAT ALAT NIP TRANSFER TO POLARIS-tithe,1.88,,263.60,NaN
5,18-May-2020,S6493060,Wema USSD Transfer from BOLU JOHNSON OGUNDOW,,"6,000.00","6,263.60",NaN
6,18-May-2020,S6689372,ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWASOGO O,"6,000.00",,263.60,NaN
7,18-May-2020,S6689372,COMM ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWAS,25.00,,238.60,NaN
8,18-May-2020,S6689372,VAT ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWASO,1.88,,236.72,NaN
9,18-May-2020,S6741605,NIP:OLUWASOGO OGUNDOWOLE-INVISIBLE TECHNOLOGIES,,"190,395.12","190,631.84",NaN


## Rename the columns properly

In [ ]:
alat.columns = ["Date", "Reference Number", "Transaction Details", "Debit", "Credit", "Balance", "Extra Column"]

## Drop the Extra Column

In [ ]:
alat = alat.drop(columns=["Extra Column"])

## drop columns where all the rows are null and reset index

In [ ]:
alat = alat.dropna(subset=["Transaction Details", "Debit", "Credit"], how="all")

alat = alat[1:].reset_index(drop=True)

In [ ]:
alat

,Date,Reference Number,Transaction Details,Debit,Credit,Balance
0,18-May-2020,M4439,Fixed Deposit Closure Proceeds - Principal,,"17,880.00","18,170.48"
1,18-May-2020,S5741873,ALAT NIP TRANSFER TO POLARIS-tithe,"17,880.00",,290.48
2,18-May-2020,S5741873,COMM ALAT NIP TRANSFER TO POLARIS-tithe,25.00,,265.48
3,18-May-2020,S5741873,VAT ALAT NIP TRANSFER TO POLARIS-tithe,1.88,,263.60
4,18-May-2020,S6493060,Wema USSD Transfer from BOLU JOHNSON OGUNDOW,,"6,000.00","6,263.60"
5,18-May-2020,S6689372,ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWASOGO O,"6,000.00",,263.60
6,18-May-2020,S6689372,COMM ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWAS,25.00,,238.60
7,18-May-2020,S6689372,VAT ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWASO,1.88,,236.72
8,18-May-2020,S6741605,NIP:OLUWASOGO OGUNDOWOLE-INVISIBLE TECHNOLOGIES,,"190,395.12","190,631.84"
9,18-May-2020,S6763049,NIP:OGUNDOWOLE OLUWASOGO OLUWAFE-,,"5,000.00","195,631.84"


## CHANGE DATE TO THE NEEDED FORMAT

In [ ]:
alat["Date"] = pd.to_datetime(alat["Date"], format="%d-%b-%Y", errors="coerce").dt.strftime("%Y-%m-%d")

## CHANGE THE FIGURES TO THE NEEDED FORMAT

In [ ]:
alat["Debit"] = alat["Debit"].replace({",": "", "":0}, regex=True).astype(float).astype(int)
alat["Credit"] = alat["Credit"].replace({",": "", "":0}, regex=True).astype(float).astype(int)
alat["Balance"] = alat["Balance"].replace({",": "", "":0}, regex=True).astype(float).astype(int)

## TRANSFORM THE ROW TO THE REQUIRED FORMAT

In [ ]:
import re

def transform_row_alat(row):
    return {
        "id": "0",
        "type": "debit" if int(row["Debit"]) > 0 else "credit",
        "amount": row["Debit"] if int(row["Debit"]) > 0 else row["Credit"],
        "narration": row["Transaction Details"].replace(r"'", ""),
        "date": row["Date"]+ "T00:00:00",
        "balance": row["Balance"] 
    }
transformed_alat = alat.apply(transform_row_alat, axis=1).tolist()
transformed_alat_df = pd.DataFrame(transformed_alat)
transformed_alat_df.head(30)

,id,type,amount,narration,date,balance
0,4439,credit,17880,Fixed Deposit Closure Proceeds - Principal,2020-05-18T00:00:00,18170
1,5741873,debit,17880,ALAT NIP TRANSFER TO POLARIS-tithe,2020-05-18T00:00:00,290
2,5741873,debit,25,COMM ALAT NIP TRANSFER TO POLARIS-tithe,2020-05-18T00:00:00,265
3,5741873,debit,1,VAT ALAT NIP TRANSFER TO POLARIS-tithe,2020-05-18T00:00:00,263
4,6493060,credit,6000,Wema USSD Transfer from BOLU JOHNSON OGUNDOW,2020-05-18T00:00:00,6263
5,6689372,debit,6000,ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWASOGO O,2020-05-18T00:00:00,263
6,6689372,debit,25,COMM ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWAS,2020-05-18T00:00:00,238
7,6689372,debit,1,VAT ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWASO,2020-05-18T00:00:00,236
8,6741605,credit,190395,NIP:OLUWASOGO OGUNDOWOLE-INVISIBLE TECHNOLOGIES,2020-05-18T00:00:00,190631
9,6763049,credit,5000,NIP:OGUNDOWOLE OLUWASOGO OLUWAFE-,2020-05-18T00:00:00,195631


## EXPORT AS JSON

In [ ]:
transformed_alat_df.to_json(orient="records")

'[{"id":"4439","type":"credit","amount":17880,"narration":"Fixed Deposit Closure Proceeds - Principal","date":"2020-05-18T00:00:00","balance":18170},{"id":"5741873","type":"debit","amount":17880,"narration":"ALAT NIP TRANSFER TO POLARIS-tithe","date":"2020-05-18T00:00:00","balance":290},{"id":"5741873","type":"debit","amount":25,"narration":"COMM ALAT NIP TRANSFER TO POLARIS-tithe","date":"2020-05-18T00:00:00","balance":265},{"id":"5741873","type":"debit","amount":1,"narration":"VAT ALAT NIP TRANSFER TO POLARIS-tithe","date":"2020-05-18T00:00:00","balance":263},{"id":"6493060","type":"credit","amount":6000,"narration":"Wema USSD Transfer from BOLU JOHNSON OGUNDOW","date":"2020-05-18T00:00:00","balance":6263},{"id":"6689372","type":"debit","amount":6000,"narration":"ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWASOGO O","date":"2020-05-18T00:00:00","balance":263},{"id":"6689372","type":"debit","amount":25,"narration":"COMM ALAT NIP TRANSFER TO GTB-OGUNDOWOLE OLUWAS","date":"2020-05-18T00:00:0